# Mission 5: Real-Data Lab — Factors on Actual Markets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/missions/blob/main/missions/mission_05_real_data/notebook.ipynb)

**Learning objectives**
- Pull real equity prices, Ken French factor data, and FRED macro series
- Run Mission 4's strategy tournament on actual market history
- Understand point-in-time (look-ahead) discipline with real macro data
- Replicate the IS/OOS factor decay documented in the Anomaly Graveyard
- Identify which factors hold up on real data vs. the synthetic market

## Part 0: Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q convexpi>=0.1.0 yfinance\n

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from convexpi.lab import (
    Backtest,
    STRATEGIES,
    compare,
    CrossSectionalMomentum,
    TimeSeriesMomentum,
    ValueTilt,
    BettingAgainstBeta,
    ICWeightedComposite,
    InverseVolatilityWeight,
    MultiFactorRank,
    TrendFilter,
)
from convexpi.lab.real_data import RealDataMarket, FrenchFactorData, FredSeries

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 100
print('Ready.')

---
## Part 1: Why Real Data?

The synthetic market from Missions 1–4 has a known structure: planted alphas, Gaussian returns, and stationary feature distributions.  Real markets are messier:

| Property | Synthetic | Real |
|---|---|---|
| Return distribution | Gaussian | Fat-tailed, skewed |
| Feature stationarity | Yes | No — factors decay |
| Survivorship bias | None | Significant |
| Macro regimes | None | Recessions, rate cycles |
| Publication bias | None | Strategies get crowded |

The **Anomaly Graveyard** page on ConvexityLabs shows this concretely.  Here's a preview of the IS vs OOS Sharpe collapse for canonical factors:

| Factor | Pre-publication Sharpe | Post-publication Sharpe | Status |
|---|---|---|---|
| Market | ~0.40 | ~0.35 | Alive |
| Size (Banz 1981) | ~0.20 | ~0.00 | Dead |
| Value (FF 1992) | ~0.35 | ~0.10 | Weakened |
| Momentum (JT 1993) | ~0.63 | ~0.38 | Weakened |
| Profitability (NM 2013) | ~0.45 | ~0.20 | Weakened |

In this mission, you'll replicate these patterns using the same data sources.

---
## Part 2: Loading Real Price Data

We use a curated universe of **50 S&P 500 constituents** with data from 2010–2024.  These are large, liquid, well-researched stocks — they have minimal survivorship bias within the window.

> **Survivorship bias warning:** By restricting to current S&P 500 members, we overweight stocks that *survived* and underweight those that were delisted or removed.  This inflates most factor backtests.  Acknowledge this in any conclusions you draw.

In [ ]:
# Universe: 50 liquid S&P 500 names across sectors
TICKERS = [
    # Technology
    'AAPL', 'MSFT', 'GOOGL', 'META', 'NVDA', 'ADBE', 'CRM', 'INTC', 'AMD', 'QCOM',
    # Financials
    'JPM', 'BAC', 'WFC', 'GS', 'MS', 'BLK', 'AXP', 'USB', 'PNC', 'COF',
    # Healthcare
    'JNJ', 'UNH', 'PFE', 'ABBV', 'MRK', 'TMO', 'DHR', 'ABT', 'BMY', 'AMGN',
    # Consumer
    'AMZN', 'HD', 'MCD', 'NKE', 'SBUX', 'TGT', 'COST', 'LOW', 'KO', 'PEP',
    # Industrials / Energy / Materials
    'XOM', 'CVX', 'BA', 'CAT', 'GE', 'MMM', 'HON', 'UPS', 'LMT', 'RTX',
]

START_DATE = '2010-01-01'
END_DATE   = '2024-01-01'

print(f"Universe: {len(TICKERS)} stocks, {START_DATE} → {END_DATE}")

In [ ]:
# Attempt to fetch real prices via yfinance
try:
    import yfinance as yf
    raw = yf.download(TICKERS, start=START_DATE, end=END_DATE,
                      auto_adjust=True, progress=False)
    prices_df = raw['Close'].dropna(how='all', axis=1).dropna(how='any', axis=0)
    prices_df = prices_df.loc[:, prices_df.isna().mean() < 0.05]  # drop sparse tickers
    USING_REAL_DATA = True
    print(f"Downloaded: {prices_df.shape[1]} stocks × {len(prices_df)} days")
    print(f"Period: {prices_df.index[0].date()} → {prices_df.index[-1].date()}")

except Exception as e:
    print(f"yfinance unavailable ({e}); using synthetic fallback.")
    USING_REAL_DATA = False

    # Reproducible synthetic proxy — same interface as the real DataFrame
    from convexpi.lab import SyntheticMarket
    _sm = SyntheticMarket(n_stocks=50, n_days=3500, seed=2010)
    _p = _sm.prices('all')
    _dates = pd.date_range('2010-01-04', periods=_p.shape[0], freq='B')
    prices_df = pd.DataFrame(_p * 100, index=_dates,
                              columns=[f'SYN{i:02d}' for i in range(_p.shape[1])])
    print(f"Synthetic fallback: {prices_df.shape[1]} stocks × {len(prices_df)} days")

prices_df.tail(3)

In [ ]:
# Quick sanity check: cumulative returns
cum_rets = (1 + prices_df.pct_change().dropna()).cumprod()
n_show = min(10, len(cum_rets.columns))

fig, ax = plt.subplots(figsize=(13, 5))
for col in cum_rets.columns[:n_show]:
    ax.plot(cum_rets.index, cum_rets[col], linewidth=0.8, alpha=0.7, label=col)
ax.set_title('Cumulative Price Returns — First 10 Stocks')
ax.set_ylabel('Cumulative return (1 = start)')
ax.legend(fontsize=7, loc='upper left')
plt.tight_layout()
plt.show()

---
## Part 3: Building the Real-Data Market

`RealDataMarket.from_prices()` converts your price DataFrame into the same interface used by `Backtest` and `compare()`.  It computes price-derived features (momentum, reversal, volatility, size) and optionally enriches them with Ken French factor data and FRED macro series.

We split the full history into:
- **Train (IS):** first 70% of days — used to develop and compare strategies
- **Test (OOS):** last 30% — reserved for final evaluation

### 3a. Price-only market (fast, works offline)

In [ ]:
market_price_only = RealDataMarket.from_prices(
    prices_df,
    train_frac=0.70,
    tc_bps=10,
    load_fred=False,
    load_french=False,
)
market_price_only.describe()
print("\nFeatures available:", sorted(market_price_only.features('train').keys()))

### 3b. Full market with Ken French + FRED (requires internet)

This adds **factor features** (Fama-French SMB, HML, MOM, Mkt-RF) and **macro features** (yield curve spread, HY credit spread, 10y/2y rates).  All are lagged by 1 business day to enforce point-in-time discipline — you can't use today's FRED release to trade today.

In [ ]:
try:
    market_full = RealDataMarket.from_prices(
        prices_df,
        train_frac=0.70,
        tc_bps=10,
        load_fred=True,
        load_french=True,
        macro_lag=1,
    )
    market_full.describe()
    print("\nFeatures available:", sorted(market_full.features('train').keys()))
    USING_FULL_MARKET = True
except Exception as e:
    print(f"Could not load French/FRED data ({e}).")
    print("Using price-only market for the rest of the notebook.")
    market_full = market_price_only
    USING_FULL_MARKET = False

# Use whichever is available
market = market_full

---
## Part 4: The Tournament on Real Data

We run the same `compare()` tournament from Mission 4, but now on actual market history.

**Important differences from the synthetic market:**
- `val_bm` and `qual_roe` are **not available** from prices alone — strategies that depend on them (ValueTilt, QualityTilt, FamaFrench3) will receive flat signals and behave like equal weight for those dimensions
- Returns are fat-tailed with real crash episodes (GFC 2008–09, COVID 2020, 2022 rate shock)
- The test split here is `'test'` not `'holdout'`

In [ ]:
# Strategies that work with price-only features
PRICE_STRATEGIES = {
    'equal_weight'   : STRATEGIES['equal_weight'],
    'momentum_12_1'  : STRATEGIES['momentum_12_1'],
    'momentum_3m'    : STRATEGIES['momentum_3m'],
    'reversal_1w'    : STRATEGIES['reversal_1w'],
    'ts_momentum'    : STRATEGIES['ts_momentum'],
    'betting_against_beta': STRATEGIES['betting_against_beta'],
    'size_premium'   : STRATEGIES['size_premium'],
    'inv_vol'        : STRATEGIES['inv_vol'],
    'min_variance'   : STRATEGIES['min_variance'],
    'trend_filter'   : STRATEGIES['trend_filter'],
    'dual_momentum'  : STRATEGIES['dual_momentum'],
}

results_real = compare(
    PRICE_STRATEGIES,
    market=market,
    split='train',
    warmup_days=252,    # 1 year warmup — 12m lookback needs it
    tc_bps=10,
)
results_real.sort_values('sharpe', ascending=False)

In [ ]:
# Side-by-side: IS Sharpe on real data vs synthetic (from Mission 4)
# (You will need to re-run Mission 4 if you want exact synthetic numbers)
fig, ax = plt.subplots(figsize=(11, 5))
sorted_r = results_real.sort_values('sharpe', ascending=True)
colors = ['#2ecc71' if s > 0 else '#e74c3c' for s in sorted_r['sharpe']]
ax.barh(sorted_r.index, sorted_r['sharpe'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('IS Sharpe Ratio (10 bps TC)')
ax.set_title('Strategy Tournament — Real Data, IS Window')
plt.tight_layout()
plt.show()

**Discussion:** Compare this bar chart to Mission 4.  Which strategies improved on real data?  Which collapsed?  What does this tell you about the synthetic market's design choices?

---
## Part 5: The Point-in-Time Trap

Many real-world backtests are inflated by **look-ahead bias** in macro data.  FRED revises historical series; GDP and inflation figures are released weeks after the reference period; yield curves are easy to pull — but only with the benefit of hindsight.

`RealDataMarket` enforces point-in-time by shifting macro features back by `macro_lag` days.  Let's show the cost of getting this wrong.

In [ ]:
if USING_FULL_MARKET:
    # Build two markets: honest 1-day lag vs. cheating (0-day lag)
    try:
        market_lag0 = RealDataMarket.from_prices(
            prices_df, train_frac=0.70, tc_bps=10,
            load_fred=True, load_french=True, macro_lag=0
        )
        market_lag1 = market_full  # already built with macro_lag=1

        bt = Backtest(warmup_days=252, tc_bps=10)
        macro_strat = MultiFactorRank(signals=['macro_yield_curve', 'mom_12m'])

        r_lag0 = bt.run(macro_strat, market_lag0.prices('train'), market_lag0.features('train'))
        r_lag1 = bt.run(macro_strat, market_lag1.prices('train'), market_lag1.features('train'))

        print(f"macro_lag=0 (biased):  Sharpe = {r_lag0.sharpe:.3f}")
        print(f"macro_lag=1 (honest):  Sharpe = {r_lag1.sharpe:.3f}")
        print(f"Look-ahead inflation:  {r_lag0.sharpe - r_lag1.sharpe:+.3f}")

    except Exception as e:
        print(f"Skipped (no FRED data): {e}")
else:
    print("Skipped: French/FRED data not loaded.")
    print("Concept: macro_lag=0 lets you trade on tomorrow's news.")
    print("Always use macro_lag >= 1 for honest backtests.")

---
## Part 6: Replicating Factor Decay

The Anomaly Graveyard tracks whether published anomalies survive in out-of-sample data.  We'll replicate the pattern for **momentum** using real market data.

The test: split the sample at a specific year (the publication cutoff), compute Sharpe before and after.  A strategy that degrades significantly post-publication suggests crowding or risk exposure rather than genuine alpha.

In [ ]:
def sharpe_in_window(strategy, market_obj, start_day, end_day, warmup=252, tc=10):
    """Compute Sharpe for a sub-window [start_day, end_day] of the full sample."""
    p = market_obj.prices('all')
    f = market_obj.features('all')
    p_window = p[start_day:end_day]
    f_window = {k: v[start_day:end_day] for k, v in f.items()}
    bt = Backtest(warmup_days=warmup, tc_bps=tc)
    return bt.run(strategy, p_window, f_window).sharpe


n = market.n_days
split = market.train_end

# Strategies with clear publication events (proxied by our IS/OOS split)
decay_strategies = {
    'CS Momentum' : CrossSectionalMomentum(),
    'TS Momentum' : TimeSeriesMomentum(),
    'BAB (low vol)': BettingAgainstBeta(),
    'Inv-Vol'     : InverseVolatilityWeight(),
    'Equal weight': STRATEGIES['equal_weight'],
}

rows = []
for name, strat in decay_strategies.items():
    is_sharpe  = sharpe_in_window(strat, market, 0,     split, warmup=252)
    oos_sharpe = sharpe_in_window(strat, market, split, n,     warmup=63)
    rows.append({'strategy': name,
                 'IS Sharpe': round(is_sharpe, 3),
                 'OOS Sharpe': round(oos_sharpe, 3),
                 'Decay': round(is_sharpe - oos_sharpe, 3)})

decay_df = pd.DataFrame(rows).set_index('strategy')
decay_df.sort_values('IS Sharpe', ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(decay_df))
w = 0.35
ax.bar(x - w/2, decay_df['IS Sharpe'],  w, label='IS',  color='steelblue', alpha=0.85)
ax.bar(x + w/2, decay_df['OOS Sharpe'], w, label='OOS', color='coral',     alpha=0.85)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(decay_df.index, rotation=15, ha='right')
ax.set_ylabel('Sharpe ratio')
ax.set_title('IS vs OOS Factor Decay — Real Market Data')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 7: Macro Regime Analysis

Many quant strategies have strong regime dependence.  Momentum is famously bad during sharp market reversals; low-volatility strategies struggle during momentum-driven bull runs.

Here we use `TrendFilter` — which switches off the inner strategy during down-trend regimes — and compare it to the raw version.

If French/FRED data is available, we also look at factor returns conditional on the yield curve regime (inverted vs. normal).

In [ ]:
bt = Backtest(warmup_days=252, tc_bps=10)

raw_mom      = CrossSectionalMomentum()
filtered_mom = TrendFilter(inner=CrossSectionalMomentum(), flat_on_down=True)

r_raw      = bt.run(raw_mom,      market.prices('train'), market.features('train'))
r_filtered = bt.run(filtered_mom, market.prices('train'), market.features('train'))

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(r_raw.cumulative_returns,      label='CS Momentum (raw)',     linewidth=1.5)
axes[0].plot(r_filtered.cumulative_returns, label='CS Momentum + TrendFilter', linewidth=1.5)
axes[0].axhline(0, color='black', lw=0.6, linestyle='--')
axes[0].set_title('Cumulative Return: Raw vs. Trend-Filtered Momentum (IS)')
axes[0].legend()

raw_dr = pd.Series(r_raw.daily_returns)
filt_dr = pd.Series(r_filtered.daily_returns)
dd_raw  = (raw_dr.cumsum() - raw_dr.cumsum().cummax())
dd_filt = (filt_dr.cumsum() - filt_dr.cumsum().cummax())
axes[1].fill_between(range(len(dd_raw)),  dd_raw,  alpha=0.5, color='steelblue', label='Raw')
axes[1].fill_between(range(len(dd_filt)), dd_filt, alpha=0.5, color='orange',    label='Filtered')
axes[1].set_title('Drawdown Profile')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Raw momentum:           Sharpe={r_raw.sharpe:.3f}  MaxDD={r_raw.max_drawdown:.2f}%  Calmar={r_raw.calmar:.3f}")
print(f"Trend-filtered momentum: Sharpe={r_filtered.sharpe:.3f}  MaxDD={r_filtered.max_drawdown:.2f}%  Calmar={r_filtered.calmar:.3f}")

---
## Part 8: Transaction Cost Reality on Real Data

In real markets, transaction costs include the bid-ask spread, market impact, and borrow cost (for shorts).  10 bps is the floor for a liquid large-cap universe; small-cap momentum strategies may face 30–50 bps.

Let's see which strategies remain viable at realistic cost levels on real data.

In [ ]:
tc_levels = [0, 5, 10, 20, 30]
focus_keys = list(PRICE_STRATEGIES.keys())

rows = []
for tc in tc_levels:
    bt_tc = Backtest(warmup_days=252, tc_bps=tc)
    for key in focus_keys:
        r = bt_tc.run(PRICE_STRATEGIES[key], market.prices('train'), market.features('train'))
        rows.append({'strategy': key, 'tc_bps': tc, 'sharpe': r.sharpe})

tc_df   = pd.DataFrame(rows)
tc_piv  = tc_df.pivot(index='strategy', columns='tc_bps', values='sharpe')
tc_piv.columns = [f'{c} bps' for c in tc_piv.columns]
tc_piv.sort_values('10 bps', ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for key in focus_keys:
    sub = tc_df[tc_df['strategy'] == key]
    ax.plot(sub['tc_bps'], sub['sharpe'], marker='o', label=key, linewidth=1.5)

ax.axhline(0, color='black', lw=0.8, linestyle='--')
ax.set_xlabel('Transaction cost (bps per side)')
ax.set_ylabel('IS Sharpe ratio')
ax.set_title('TC Sensitivity on Real Data')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()

---
## Part 9: The Final Evaluation

You've been working only in the IS window.  Now look at the OOS window **once** to see what survives.

Remember: the OOS window is the last 30% of the sample — this is a different time period than the IS window, not a different random draw.  If IS was 2010–2019, OOS is 2019–2024.

In [ ]:
results_oos = compare(
    PRICE_STRATEGIES,
    market=market,
    split='test',       # NOTE: RealDataMarket uses 'test', not 'holdout'
    warmup_days=63,
    tc_bps=10,
)

final = pd.DataFrame({
    'IS Sharpe' : results_real['sharpe'],
    'OOS Sharpe': results_oos['sharpe'],
    'IS Return' : results_real['annual_return'].map('{:.1f}%'.format),
    'OOS Return': results_oos['annual_return'].map('{:.1f}%'.format),
    'Turnover'  : results_real['annual_turnover'].map('{:.1f}x'.format),
}).sort_values('IS Sharpe', ascending=False)

final

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(final))
w = 0.35
ax.bar(x - w/2, final['IS Sharpe'],  w, label='IS',  color='steelblue', alpha=0.85)
ax.bar(x + w/2, final['OOS Sharpe'], w, label='OOS', color='coral',     alpha=0.85)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(final.index, rotation=20, ha='right')
ax.set_ylabel('Sharpe ratio')
ax.set_title('Real Data: IS vs OOS Sharpe by Strategy')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 10: Challenges

Complete any **two** of the following.

### Challenge A: Expand the Universe

The 50-stock universe above excludes smaller companies, international markets, and sector ETFs.  Try one of:
- Replace TICKERS with sector ETFs (`XLK`, `XLF`, `XLE`, `XLV`, `XLI`, `XLY`, `XLP`, `XLU`, `XLRE`, `XLB`)
- Add Russell 2000 components (warning: survivorship bias is severe for small caps)
- Use country ETFs (`EEM`, `VGK`, `EWJ`, `EWC`, `EWA`) — a multi-country momentum test

Does your chosen universe change which strategies survive OOS?

In [ ]:
# Challenge A — your solution here
ALT_TICKERS = ['XLK', 'XLF', 'XLE', 'XLV', 'XLI', 'XLY', 'XLP', 'XLU', 'XLRE', 'XLB']

try:
    import yfinance as yf
    raw_alt = yf.download(ALT_TICKERS, start='2000-01-01', end='2024-01-01',
                          auto_adjust=True, progress=False)
    prices_alt = raw_alt['Close'].dropna(how='any', axis=0)
    market_alt  = RealDataMarket.from_prices(prices_alt, train_frac=0.70,
                                              load_fred=False, load_french=False)
    results_alt = compare(PRICE_STRATEGIES, market=market_alt, split='train',
                          warmup_days=252, tc_bps=10)
    print(results_alt.sort_values('sharpe', ascending=False)[['sharpe','annual_return','annual_turnover']])
except Exception as e:
    print(f'Skipped: {e}')

### Challenge B: The IC Audit

For each signal available in the real-data market, compute the rolling 63-day Information Coefficient (rank correlation between the signal on day $t$ and the 1-day return on day $t+1$).  Plot the IC over time.

**Questions to answer:**
1. Which signal has the most stable IC?
2. Is there evidence of IC decay over time (publication effect)?
3. Does IC volatility explain why the `ICWeightedComposite` underperforms static factor weights in some windows?

In [ ]:
# Challenge B — your solution here
from scipy.stats import spearmanr

prices = market.prices('train')
features = market.features('train')
T = prices.shape[0]

returns_1d = prices[1:] / prices[:-1] - 1  # shape (T-1, N)

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
axes = axes.flatten()

signal_keys = ['mom_12m', 'mom_3m', 'reversal_1w', 'vol_1m', 'size_cap', 'mom_1m']

for ax, key in zip(axes, signal_keys):
    sig = features.get(key)
    if sig is None:
        ax.set_visible(False)
        continue

    ic_series = []
    for t in range(63, T - 1):
        s = sig[t]
        r = returns_1d[t]
        mask = np.isfinite(s) & np.isfinite(r)
        if mask.sum() > 10:
            ic, _ = spearmanr(s[mask], r[mask])
        else:
            ic = np.nan
        ic_series.append(ic)

    ic_rolling = pd.Series(ic_series).rolling(63).mean()
    ax.plot(ic_rolling.values, linewidth=1, color='steelblue')
    ax.axhline(0, color='black', lw=0.8, linestyle='--')
    ax.set_title(f'{key}  (63d rolling IC)')
    ax.set_ylabel('IC')

plt.suptitle('Rolling 63-day IC by Signal — Real Market Data', y=1.01)
plt.tight_layout()
plt.show()

### Challenge C: Build a Real-Data Grader Submission

The platform grader uses `SyntheticMarket` by default.  But you can benchmark your real-data strategy against the same leaderboard by wrapping it as a standard `Strategy` subclass.

Build a strategy that uses signals from the real-data market that also exist in the synthetic market (`mom_12m`, `mom_3m`, `reversal_1w`, `vol_1m`, `size_cap`).  Submit it to the grader — it should work on both market types without modification.

In [ ]:
# Challenge C — your solution here
from convexpi.lab import Strategy, SyntheticMarket

class RealDataInspiredStrategy(Strategy):
    """
    A strategy derived from real-data analysis that only uses
    features available in both RealDataMarket and SyntheticMarket.
    Compatible with the platform grader.
    """
    def on_day(self, day, features, prices, portfolio):
        n = len(prices)
        # Your implementation — use only: mom_12m, mom_3m, reversal_1w, vol_1m, size_cap
        return np.zeros(n)

# Validate on both market types
bt = Backtest(warmup_days=252, tc_bps=10)

print('--- Real-data market ---')
r_real = bt.run(RealDataInspiredStrategy(), market.prices('train'), market.features('train'))
r_real.tearsheet()

print('--- Synthetic market (grader-compatible) ---')
sm = SyntheticMarket(n_stocks=200, n_days=1000, seed=42)
bt_synth = Backtest(warmup_days=63, tc_bps=10)
r_synth = bt_synth.run(RealDataInspiredStrategy(), sm.prices('train'), sm.features('train'))
r_synth.tearsheet()

---
## Wrap-up

**What you covered:**

| Concept | Where |
|---|---|
| Real vs. synthetic market differences (fat tails, survivorship, regime) | Part 1 |
| Loading prices via yfinance + fallback to synthetic | Part 2 |
| Building `RealDataMarket` with French/FRED enrichment | Part 3 |
| Strategy tournament on actual history | Part 4 |
| Point-in-time (look-ahead) enforcement via `macro_lag` | Part 5 |
| Factor decay: IS vs OOS Sharpe pattern | Part 6 |
| Regime analysis with `TrendFilter` | Part 7 |
| Transaction cost reality on real data | Part 8 |
| Final IS vs OOS evaluation | Part 9 |

**Key takeaways:**

1. **Survivorship bias inflates every backtest** on a universe of current index members. This is unavoidable in most academic exercises — acknowledge it explicitly.
2. **Point-in-time discipline matters.** Even a 1-day macro lag changes results. In production, data publication lags can be 30+ days.
3. **Factors decay post-publication.** The Anomaly Graveyard documents this across 50+ factors. Momentum and value are weakened but not dead; size is effectively dead on large-cap universes.
4. **Real return distributions are fat-tailed.** Sharpe ratios computed over windows including 2008, 2020, or 2022 will look different from windows that don't.
5. **A strategy that works on synthetic data may fail on real data** due to regime changes, factor crowding, and macro exposure the synthetic model doesn't capture.